# C-MAPSS FD001 Mini-Project

This notebook trains two models for the 24-788 solo mini-project:

- **Baseline:** LSTM
- **Variant:** TCN

It downloads the NASA C-MAPSS FD001 dataset, trains both models, saves checkpoints, and regenerates plots.


In [17]:
# Optional: mount Drive so outputs persist
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [19]:
# Install dependencies
!pip -q install numpy pandas matplotlib scikit-learn torch

In [20]:
# Create project folder
import os, pathlib
PROJECT_DIR = pathlib.Path('/content/cmapss_project')
PROJECT_DIR.mkdir(exist_ok=True)
os.chdir(PROJECT_DIR)
print(PROJECT_DIR)

/content/cmapss_project


In [21]:
# Manual, robust C-MAPSS FD001 download + extraction
from pathlib import Path
import shutil
import urllib.request
import zipfile
import os

data_dir = Path("data/CMAPSS")
data_dir.mkdir(parents=True, exist_ok=True)

zip_path = data_dir / "CMAPSSData.zip"
url = "https://data.nasa.gov/docs/legacy/CMAPSSData.zip"

print("Downloading:", url)
urllib.request.urlretrieve(url, zip_path)
print("Saved to:", zip_path)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(data_dir)

wanted = ["train_FD001.txt", "test_FD001.txt", "RUL_FD001.txt"]

for fname in wanted:
    matches = list(data_dir.rglob(fname))
    print(fname, "matches:", matches[:5])
    if not matches:
        raise FileNotFoundError(f"Could not find {fname} anywhere under {data_dir}")
    target = data_dir / fname
    if matches[0] != target:
        shutil.copy(matches[0], target)

print({f: (data_dir / f).exists() for f in wanted})

Downloading: https://data.nasa.gov/docs/legacy/CMAPSSData.zip
Saved to: data/CMAPSS/CMAPSSData.zip
train_FD001.txt matches: [PosixPath('data/CMAPSS/train_FD001.txt')]
test_FD001.txt matches: [PosixPath('data/CMAPSS/test_FD001.txt')]
RUL_FD001.txt matches: [PosixPath('data/CMAPSS/RUL_FD001.txt')]
{'train_FD001.txt': True, 'test_FD001.txt': True, 'RUL_FD001.txt': True}


In [22]:
# Write starter files into Colab
from pathlib import Path

files = {
    'train.py': r'''
import argparse
import json
import math
import os
import random
import shutil
import urllib.request
import zipfile
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
DATA_URL = "https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip"

COL_NAMES = (
    ["unit", "cycle"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def phm_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    d = y_pred - y_true
    score = np.where(d < 0, np.exp(-d / 13.0) - 1.0, np.exp(d / 10.0) - 1.0)
    return float(np.sum(score))

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def maybe_download_dataset(data_dir: Path) -> None:
    data_dir.mkdir(parents=True, exist_ok=True)
    train_file = data_dir / "train_FD001.txt"
    test_file = data_dir / "test_FD001.txt"
    rul_file = data_dir / "RUL_FD001.txt"
    if train_file.exists() and test_file.exists() and rul_file.exists():
        print("Dataset already exists.")
        return

    zip_path = data_dir / "cmapss.zip"
    print(f"Downloading dataset from {DATA_URL} ...")
    urllib.request.urlretrieve(DATA_URL, zip_path)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(data_dir)

    extracted_root = data_dir / "6. Turbofan Engine Degradation Simulation Data Set"
    if extracted_root.exists():
        for fname in ["train_FD001.txt", "test_FD001.txt", "RUL_FD001.txt"]:
            src = extracted_root / fname
            dst = data_dir / fname
            if src.exists() and not dst.exists():
                shutil.copy(src, dst)
    print("Dataset ready.")

def load_cmapss_split(data_dir: Path, split: str) -> pd.DataFrame:
    if split == "train":
        path = data_dir / "train_FD001.txt"
    elif split == "test":
        path = data_dir / "test_FD001.txt"
    else:
        raise ValueError("split must be train or test")

    df = pd.read_csv(path, sep=r"\s+", header=None)
    # Some distributions contain 26 real columns + 2 empty columns at the end.
    if df.shape[1] > 26:
        df = df.iloc[:, :26]
    df.columns = COL_NAMES
    return df

def load_rul_truth(data_dir: Path) -> pd.DataFrame:
    path = data_dir / "RUL_FD001.txt"
    rul = pd.read_csv(path, sep=r"\s+", header=None)
    rul = rul.iloc[:, :1]
    rul.columns = ["RUL"]
    rul["unit"] = np.arange(1, len(rul) + 1)
    return rul

def compute_train_rul(train_df: pd.DataFrame, max_rul: int) -> pd.DataFrame:
    max_cycle = train_df.groupby("unit")["cycle"].max().rename("max_cycle")
    out = train_df.merge(max_cycle, on="unit")
    out["RUL"] = out["max_cycle"] - out["cycle"]
    out["RUL"] = out["RUL"].clip(upper=max_rul)
    return out.drop(columns=["max_cycle"])

def choose_features(train_df: pd.DataFrame, std_threshold: float = 1e-8):
    feature_cols = [c for c in train_df.columns if c.startswith("op_setting_") or c.startswith("sensor_")]
    stds = train_df[feature_cols].std()
    keep = stds[stds > std_threshold].index.tolist()
    return keep

@dataclass
class PreparedData:
    train_df: pd.DataFrame
    val_df: pd.DataFrame
    test_df: pd.DataFrame
    test_rul: pd.DataFrame
    feature_cols: list
    scaler: StandardScaler

def split_by_unit(train_df: pd.DataFrame, val_frac: float = 0.2, seed: int = SEED):
    units = sorted(train_df["unit"].unique())
    rng = np.random.default_rng(seed)
    rng.shuffle(units)
    n_val = max(1, int(len(units) * val_frac))
    val_units = set(units[:n_val])
    train_units = set(units[n_val:])
    tr = train_df[train_df["unit"].isin(train_units)].copy()
    va = train_df[train_df["unit"].isin(val_units)].copy()
    return tr, va

def prepare_data(data_dir: Path, max_rul: int = 125, val_frac: float = 0.2) -> PreparedData:
    train_df = load_cmapss_split(data_dir, "train")
    test_df = load_cmapss_split(data_dir, "test")
    test_rul = load_rul_truth(data_dir)

    train_df = compute_train_rul(train_df, max_rul=max_rul)

    # For test, RUL is only defined at the end of each unit. We keep raw test_df and use truth later.
    feature_cols = choose_features(train_df)

    tr_df, va_df = split_by_unit(train_df, val_frac=val_frac, seed=SEED)

    scaler = StandardScaler()
    scaler.fit(tr_df[feature_cols])

    for df in [tr_df, va_df, test_df]:
        df.loc[:, feature_cols] = scaler.transform(df[feature_cols])

    return PreparedData(
        train_df=tr_df,
        val_df=va_df,
        test_df=test_df,
        test_rul=test_rul,
        feature_cols=feature_cols,
        scaler=scaler,
    )

class SequenceWindowDataset(Dataset):
    def __init__(self, df: pd.DataFrame, feature_cols: list, window_size: int, stride: int = 1, train_mode: bool = True):
        self.samples = []
        self.feature_cols = feature_cols
        self.window_size = window_size
        self.train_mode = train_mode

        for unit_id, unit_df in df.groupby("unit"):
            unit_df = unit_df.sort_values("cycle").reset_index(drop=True)
            features = unit_df[feature_cols].values.astype(np.float32)

            if train_mode:
                targets = unit_df["RUL"].values.astype(np.float32)
                if len(unit_df) < window_size:
                    pad_len = window_size - len(unit_df)
                    pad_feats = np.repeat(features[:1], pad_len, axis=0)
                    x = np.concatenate([pad_feats, features], axis=0)
                    y = targets[-1]
                    self.samples.append((x, y))
                else:
                    for end in range(window_size - 1, len(unit_df), stride):
                        start = end - window_size + 1
                        x = features[start : end + 1]
                        y = targets[end]
                        self.samples.append((x, y))
            else:
                # One sample per engine: final window only
                if len(unit_df) < window_size:
                    pad_len = window_size - len(unit_df)
                    pad_feats = np.repeat(features[:1], pad_len, axis=0)
                    x = np.concatenate([pad_feats, features], axis=0)
                else:
                    x = features[-window_size:]
                self.samples.append((unit_id, x))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        if self.train_mode:
            x, y = item
            return torch.from_numpy(x), torch.tensor(y, dtype=torch.float32)
        unit_id, x = item
        return unit_id, torch.from_numpy(x)

class LSTMRegressor(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 64, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        pred = self.head(last).squeeze(-1)
        return pred

class Chomp1d(nn.Module):
    def __init__(self, chomp_size: int):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous() if self.chomp_size > 0 else x

class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_ch, out_ch, kernel_size, padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCNRegressor(nn.Module):
    def __init__(self, input_dim: int, channels=(64, 64, 64), kernel_size: int = 3, dropout: float = 0.2):
        super().__init__()
        layers = []
        in_ch = input_dim
        for i, out_ch in enumerate(channels):
            layers.append(TemporalBlock(in_ch, out_ch, kernel_size=kernel_size, dilation=2**i, dropout=dropout))
            in_ch = out_ch
        self.tcn = nn.Sequential(*layers)
        self.head = nn.Sequential(
            nn.Linear(channels[-1], 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        # x: [B, T, F] -> [B, F, T]
        x = x.transpose(1, 2)
        feat = self.tcn(x)
        last = feat[:, :, -1]
        pred = self.head(last).squeeze(-1)
        return pred

def make_model(model_name: str, input_dim: int):
    if model_name == "lstm":
        return LSTMRegressor(input_dim=input_dim)
    if model_name == "tcn":
        return TCNRegressor(input_dim=input_dim)
    raise ValueError("model must be lstm or tcn")

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    n = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        batch_n = x.size(0)
        total_loss += loss.item() * batch_n
        n += batch_n
    return total_loss / max(1, n)

@torch.no_grad()
def evaluate_regression(model, loader, device):
    model.eval()
    ys, preds = [], []
    for x, y in loader:
        x = x.to(device)
        pred = model(x).cpu().numpy()
        preds.append(pred)
        ys.append(y.numpy())
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(preds)
    return {
        "rmse": rmse(y_true, y_pred),
        "phm_score": phm_score(y_true, y_pred),
        "y_true": y_true.tolist(),
        "y_pred": y_pred.tolist(),
    }

@torch.no_grad()
def evaluate_test_last_window(model, dataset: SequenceWindowDataset, test_rul: pd.DataFrame, device):
    model.eval()
    unit_ids, preds = [], []
    for unit_id, x in dataset:
        x = x.unsqueeze(0).to(device)
        pred = model(x).item()
        unit_ids.append(unit_id)
        preds.append(pred)

    pred_df = pd.DataFrame({"unit": unit_ids, "pred_rul": preds})
    merged = pred_df.merge(test_rul, on="unit", how="inner")
    y_true = merged["RUL"].values.astype(float)
    y_pred = merged["pred_rul"].values.astype(float)
    return {
        "rmse": rmse(y_true, y_pred),
        "phm_score": phm_score(y_true, y_pred),
        "y_true": y_true.tolist(),
        "y_pred": y_pred.tolist(),
    }

def plot_learning_curves(history: dict, output_path: Path):
    plt.figure(figsize=(7, 4))
    plt.plot(history["train_loss"], label="Train MSE")
    plt.plot(history["val_rmse"], label="Val RMSE")
    plt.xlabel("Epoch")
    plt.ylabel("Metric")
    plt.title("Training Curve")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()

def plot_scatter(y_true, y_pred, output_path: Path):
    plt.figure(figsize=(5, 5))
    plt.scatter(y_true, y_pred, alpha=0.7)
    lo = min(min(y_true), min(y_pred))
    hi = max(max(y_true), max(y_pred))
    plt.plot([lo, hi], [lo, hi], linestyle="--")
    plt.xlabel("True RUL")
    plt.ylabel("Predicted RUL")
    plt.title("True vs Predicted RUL")
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", type=str, default="data/CMAPSS")
    parser.add_argument("--output_dir", type=str, required=True)
    parser.add_argument("--model", type=str, choices=["lstm", "tcn"], required=True)
    parser.add_argument("--window_size", type=int, default=30)
    parser.add_argument("--max_rul", type=int, default=125)
    parser.add_argument("--epochs", type=int, default=30)
    parser.add_argument("--batch_size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--val_frac", type=float, default=0.2)
    parser.add_argument("--patience", type=int, default=8)
    parser.add_argument("--stride", type=int, default=1)
    args = parser.parse_args()

    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    data_dir = Path(args.data_dir)
    maybe_download_dataset(data_dir)
    prepared = prepare_data(data_dir, max_rul=args.max_rul, val_frac=args.val_frac)

    train_ds = SequenceWindowDataset(prepared.train_df, prepared.feature_cols, window_size=args.window_size, stride=args.stride, train_mode=True)
    val_ds = SequenceWindowDataset(prepared.val_df, prepared.feature_cols, window_size=args.window_size, stride=1, train_mode=True)
    test_ds = SequenceWindowDataset(prepared.test_df, prepared.feature_cols, window_size=args.window_size, train_mode=False)

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, drop_last=False)

    model = make_model(args.model, input_dim=len(prepared.feature_cols)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    loss_fn = nn.MSELoss()

    history = {"train_loss": [], "val_rmse": [], "val_phm_score": []}
    best_state = None
    best_val_rmse = math.inf
    patience_counter = 0

    for epoch in range(1, args.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        val_metrics = evaluate_regression(model, val_loader, device)

        history["train_loss"].append(train_loss)
        history["val_rmse"].append(val_metrics["rmse"])
        history["val_phm_score"].append(val_metrics["phm_score"])

        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={train_loss:.4f} | "
            f"val_rmse={val_metrics['rmse']:.4f} | "
            f"val_phm={val_metrics['phm_score']:.2f}"
        )

        if val_metrics["rmse"] < best_val_rmse:
            best_val_rmse = val_metrics["rmse"]
            best_state = {
                "model_state_dict": model.state_dict(),
                "feature_cols": prepared.feature_cols,
                "window_size": args.window_size,
                "max_rul": args.max_rul,
                "model_name": args.model,
            }
            torch.save(best_state, output_dir / "best_model.pt")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= args.patience:
                print("Early stopping triggered.")
                break

    checkpoint = torch.load(output_dir / "best_model.pt", map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    val_metrics = evaluate_regression(model, val_loader, device)
    test_metrics = evaluate_test_last_window(model, test_ds, prepared.test_rul, device)

    with open(output_dir / "history.json", "w") as f:
        json.dump(history, f, indent=2)

    with open(output_dir / "metrics.json", "w") as f:
        json.dump({"val": val_metrics, "test": test_metrics}, f, indent=2)

    plot_learning_curves(history, output_dir / "learning_curve.png")
    plot_scatter(test_metrics["y_true"], test_metrics["y_pred"], output_dir / "test_scatter.png")

    summary = {
        "model": args.model,
        "device": str(device),
        "window_size": args.window_size,
        "feature_count": len(prepared.feature_cols),
        "feature_cols": prepared.feature_cols,
        "val_rmse": val_metrics["rmse"],
        "val_phm_score": val_metrics["phm_score"],
        "test_rmse": test_metrics["rmse"],
        "test_phm_score": test_metrics["phm_score"],
    }
    with open(output_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("\nTraining complete.")
    print(json.dumps(summary, indent=2))

if __name__ == "__main__":
    main()
''',
    'reproduce_results.py': r'''
import argparse
import json
from pathlib import Path

import torch
from train import (
    SequenceWindowDataset,
    evaluate_regression,
    evaluate_test_last_window,
    make_model,
    maybe_download_dataset,
    plot_learning_curves,
    plot_scatter,
    prepare_data,
)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--run_dir", type=str, required=True)
    parser.add_argument("--data_dir", type=str, default="data/CMAPSS")
    parser.add_argument("--val_frac", type=float, default=0.2)
    args = parser.parse_args()

    run_dir = Path(args.run_dir)
    checkpoint = torch.load(run_dir / "best_model.pt", map_location="cpu")

    maybe_download_dataset(Path(args.data_dir))
    prepared = prepare_data(Path(args.data_dir), max_rul=checkpoint["max_rul"], val_frac=args.val_frac)

    val_ds = SequenceWindowDataset(prepared.val_df, prepared.feature_cols, window_size=checkpoint["window_size"], train_mode=True)
    test_ds = SequenceWindowDataset(prepared.test_df, prepared.feature_cols, window_size=checkpoint["window_size"], train_mode=False)

    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=256, shuffle=False)

    model = make_model(checkpoint["model_name"], input_dim=len(prepared.feature_cols))
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    val_metrics = evaluate_regression(model, val_loader, device="cpu")
    test_metrics = evaluate_test_last_window(model, test_ds, prepared.test_rul, device="cpu")

    history_path = run_dir / "history.json"
    if history_path.exists():
        history = json.loads(history_path.read_text())
        plot_learning_curves(history, run_dir / "learning_curve_reproduced.png")

    plot_scatter(test_metrics["y_true"], test_metrics["y_pred"], run_dir / "test_scatter_reproduced.png")

    out = {"val": val_metrics, "test": test_metrics}
    with open(run_dir / "reproduced_metrics.json", "w") as f:
        json.dump(out, f, indent=2)

    print(json.dumps(out, indent=2))

if __name__ == "__main__":
    main()
''',
    'requirements.txt': r'''numpy>=1.24
pandas>=2.0
matplotlib>=3.7
scikit-learn>=1.3
torch>=2.1
''',
    'README.md': r'''# C-MAPSS FD001 Mini-Project Starter

This starter package is for the 24-788 mini-project using the NASA C-MAPSS FD001 dataset.

## Models
- Baseline: LSTM regressor
- Variant: Temporal Convolutional Network (TCN) regressor

## Files
- `train.py` — trains one model (`lstm` or `tcn`)
- `reproduce_results.py` — reloads a saved checkpoint and regenerates metrics/plots
- `requirements.txt` — minimal Python dependencies
- `cmapss_colab_starter.ipynb` — Colab notebook wrapper around the scripts

## Quick start
```bash
pip install -r requirements.txt
python train.py --model lstm --window_size 30 --epochs 30 --batch_size 128 --output_dir runs/lstm
python train.py --model tcn --window_size 30 --epochs 30 --batch_size 128 --output_dir runs/tcn
python reproduce_results.py --run_dir runs/lstm
python reproduce_results.py --run_dir runs/tcn
```

## Recommended project settings
- Dataset subset: FD001 only
- Target clipping: 125
- Window size: 30
- Validation split: split by engine unit, not by windows
- Metrics: RMSE and PHM score
''',
}
for name, content in files.items():
    Path(name).write_text(content)
print('Files written:', list(files.keys()))

Files written: ['train.py', 'reproduce_results.py', 'requirements.txt', 'README.md']


In [23]:
# Train baseline LSTM
!python train.py --model lstm --window_size 30 --epochs 30 --batch_size 128 --output_dir runs/lstm

Dataset already exists.
/content/cmapss_project/train.py:141: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.78083775 -0.78083775 -2.07234315 ...  1.15642036  1.15642036
  2.44792576]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[:, feature_cols] = scaler.transform(df[feature_cols])
/content/cmapss_project/train.py:141: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.42659045 -2.07234315 -0.13508505 ...  3.09367846  1.15642036
  1.80217306]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[:, feature_cols] = scaler.transform(df[feature_cols])
/content/cmapss_project/train.py:141: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.78083775 -0.13508505 -0.13508505 ...  1.1564

In [24]:
# Train variant TCN
!python train.py --model tcn --window_size 30 --epochs 30 --batch_size 128 --output_dir runs/tcn

Dataset already exists.
/content/cmapss_project/train.py:141: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.78083775 -0.78083775 -2.07234315 ...  1.15642036  1.15642036
  2.44792576]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[:, feature_cols] = scaler.transform(df[feature_cols])
/content/cmapss_project/train.py:141: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.42659045 -2.07234315 -0.13508505 ...  3.09367846  1.15642036
  1.80217306]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[:, feature_cols] = scaler.transform(df[feature_cols])
/content/cmapss_project/train.py:141: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.78083775 -0.13508505 -0.13508505 ...  1.1564

In [25]:
# Reproduce outputs from saved checkpoints
!python reproduce_results.py --run_dir runs/lstm
!python reproduce_results.py --run_dir runs/tcn

Streaming output truncated to the last 5000 lines.
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      125.0,
      124.0,
      123.0,
      122.0,
      121.0,
      120.0,
      119.0,
      118.0,
      117.0,
      116.0,
      115.0,
      114.0,
      113.0,
      112.0,
      111.0,
      110.0,
      109.0,
      108.0,
      107.0,
      106.0,
      105.0,
      104.0,
      103.0,
      102.0,
      101.0,
      100.0,
      99.0,
      98.0,
      97.0,
      96.0,
      95.0,
      94.0,
      93.0,
      92.0,
      91.0,
      90.0,
      89.0,
      88.0,
      87.0,
      86.0,
      85.0,
      84.0,
      83.0,
      82.0,
      81.0,
      80.0,
      7

In [26]:
# Compare results
import json
from pathlib import Path

def load_summary(path):
    return json.loads(Path(path).read_text())

lstm = load_summary('runs/lstm/summary.json')
tcn = load_summary('runs/tcn/summary.json')

print('LSTM test RMSE:', lstm['test_rmse'])
print('LSTM test PHM score:', lstm['test_phm_score'])
print('TCN test RMSE:', tcn['test_rmse'])
print('TCN test PHM score:', tcn['test_phm_score'])

LSTM test RMSE: 14.891303090365625
LSTM test PHM score: 369.19099797090996
TCN test RMSE: 17.45688909847654
TCN test PHM score: 542.4512111909736


In [27]:
# Copy outputs to Drive if we mounted it
!mkdir -p /content/drive/MyDrive/cmapss_project_outputs
!cp -r runs /content/drive/MyDrive/cmapss_project_outputs/
print('Copied runs/ to Drive')

Copied runs/ to Drive
